In [1]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import sqlite3

# 1. Load the messy dataset
df = pd.read_csv('QuickHyre_Messy_Job_Postings.csv')

# 2. Create a local SQL database on your computer
conn = sqlite3.connect('quickhyre_jobs.db')
cursor = conn.cursor()

# 3. Push the messy data into a SQL table called 'raw_jobs'
df.to_sql('raw_jobs', conn, if_exists='replace', index=False)

print("Messy data successfully loaded into SQL database!")

Messy data successfully loaded into SQL database!


In [3]:
# Query 1: Data Cleaning & Standardization using SQL
cleaning_query = """
WITH CleanedData AS (
    SELECT 
        job_id,
        -- Clean job titles: uppercase it and trim white space
        TRIM(UPPER(raw_job_title)) AS clean_title,
        company_name,
        -- Extract the minimum salary using CASE and string manipulation
        CASE 
            WHEN salary_range LIKE '$%k - $%k' THEN 
                CAST(SUBSTR(salary_range, 2, INSTR(salary_range, 'k') - 2) AS INTEGER) * 1000
            WHEN salary_range LIKE '%-%' AND salary_range NOT LIKE '$%' THEN 
                CAST(SUBSTR(salary_range, 1, INSTR(salary_range, '-') - 1) AS INTEGER)
            ELSE NULL 
        END AS min_salary_usd,
        -- Remove basic HTML tags from the description (simulate Regex in SQL)
        REPLACE(REPLACE(REPLACE(job_description_html, '<p>', ''), '</p>', ''), '<ul>', '') AS clean_desc
    FROM raw_jobs
    WHERE posted_date IS NOT NULL
)

-- Use a Window Function to rank salaries within each company
SELECT 
    clean_title,
    company_name,
    min_salary_usd,
    RANK() OVER(PARTITION BY company_name ORDER BY min_salary_usd DESC) as salary_rank
FROM CleanedData
WHERE min_salary_usd IS NOT NULL
LIMIT 15;
"""

# Execute and display the results
cleaned_df = pd.read_sql_query(cleaning_query, conn)
display(cleaned_df)

,clean_title,company_name,min_salary_usd,salary_rank
0,DATA ANALYST - SQL,DataInc,90000,1
1,DATA ANALYST,DataInc,90000,1
2,SR. DATA ANALYST,DataInc,89969,3
3,DATA ANALYST,DataInc,89905,4
4,SR. DATA ANALYST,DataInc,89728,5
5,DATA ANALYST,DataInc,89130,6
6,DATA ANALYST - SQL,DataInc,89098,7
7,LEAD ANALYTICS,DataInc,89000,8
8,DATA ANALYST,DataInc,89000,8
9,LEAD ANALYTICS,DataInc,89000,8


In [4]:
skill_query = """
SELECT 
    SUM(CASE WHEN job_description_html LIKE '%Python%' THEN 1 ELSE 0 END) AS Python_Count,
    SUM(CASE WHEN job_description_html LIKE '%SQL%' THEN 1 ELSE 0 END) AS SQL_Count,
    SUM(CASE WHEN job_description_html LIKE '%Power BI%' THEN 1 ELSE 0 END) AS PowerBI_Count,
    SUM(CASE WHEN job_description_html LIKE '%Tableau%' THEN 1 ELSE 0 END) AS Tableau_Count
FROM raw_jobs;
"""

skills_df = pd.read_sql_query(skill_query, conn)
print("\n--- Most Demanded Skills in the Market ---")
display(skills_df)


--- Most Demanded Skills in the Market ---


,Python_Count,SQL_Count,PowerBI_Count,Tableau_Count
0,643,763,638,660
